In [ ]:
import random
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker
from sqlalchemy import create_engine
import urllib.parse

fake = Faker()
Faker.seed(42)
random.seed(42)

SQL_SERVER = "YOURPC\\SQLEXPRESS"   
DATABASE = "OpsExceptionsDB"           
TABLE_NAME = "Exceptions"

USE_WINDOWS_AUTH = True

#or

SQL_USERNAME = "Rashmi"
SQL_PASSWORD = "your_password_here"  # only used if USE_WINDOWS_AUTH is False

ROW_COUNT = 2000

CATEGORIES = [
    "SLA Breach",
    "Data Error",
    "System Downtime",
    "Process Deviation",
    "Compliance Issue",
    "Billing Discrepancy",
    "Vendor Delay",
]

SEVERITIES = ["Low", "Medium", "High"]

SEVERITY_WEIGHTS = [0.5, 0.35, 0.15]

STATUSES = ["Pending", "Approved", "Rejected", "Closed/Resolved"]

OWNERS = [fake.name() for _ in range(15)]  
APPROVERS = [fake.name() for _ in range(4)] 

APPROVER_COMMENTS_APPROVED = [
    "Reviewed and approved. Root cause documented.",
    "Approved — standard corrective action applied.",
    "Approved after confirming with vendor.",
    "Approved. No recurrence expected.",
]
APPROVER_COMMENTS_REJECTED = [
    "Needs more detail on root cause before approval.",
    "Rejected — please attach supporting evidence.",
    "Rejected. Escalate to vendor management instead.",
    "Insufficient justification for severity level.",
]

DESCRIPTIONS_BY_CATEGORY = {
    "SLA Breach": [
        "Turnaround time for client request exceeded the agreed 24-hour SLA by 6 hours.",
        "Ticket resolution SLA breached due to backlog during shift handover.",
        "Response time to escalation exceeded contractual SLA threshold.",
        "End-of-day report was delivered 2 hours past the committed SLA window.",
    ],
    "Data Error": [
        "Customer account records contained duplicate entries following the latest system upload.",
        "Incorrect billing amount identified during the daily reconciliation process.",
        "Mismatched transaction totals found between source system and reporting extract.",
        "Data field mapping error caused incorrect category tagging on 40+ records.",
    ],
    "System Downtime": [
        "Operations portal was unavailable for 42 minutes, affecting transaction processing.",
        "Core reporting database experienced an unplanned outage during business hours.",
        "Scheduled batch job failed to run due to a server connectivity issue overnight.",
        "Intermittent system slowness reported by multiple users during peak processing.",
    ],
    "Process Deviation": [
        "Required approval step was bypassed during the manual processing of the transaction.",
        "Standard verification checklist was not completed before case closure.",
        "Handoff between shifts did not follow the documented escalation procedure.",
        "Transaction was processed outside the defined four-eyes review policy.",
    ],
    "Compliance Issue": [
        "Required documentation was missing from the client onboarding file.",
        "Retention policy was not applied correctly to archived client records.",
        "Transaction flagged for review lacked the mandatory compliance sign-off.",
        "Audit sample identified a gap in KYC documentation for one account.",
    ],
    "Billing Discrepancy": [
        "Invoice amount did not match the agreed contract rate for the billing period.",
        "Duplicate charge applied to client account during monthly billing run.",
        "Currency conversion error resulted in an incorrect invoice total.",
        "Credit note was not applied before the invoice was issued to the client.",
    ],
    "Vendor Delay": [
        "Third-party data feed was delivered 5 hours later than the agreed schedule.",
        "Vendor system maintenance caused a delay in downstream report generation.",
        "Outsourced processing partner missed the agreed same-day turnaround.",
        "Delayed vendor confirmation held up settlement processing for the batch.",
    ],
}


RESOLUTION_NOTES_BY_CATEGORY = {
    "SLA Breach": "SLA breach reviewed with team lead; workload redistributed to prevent recurrence.",
    "Data Error": "Affected records identified and corrected; reconciliation re-run to confirm accuracy.",
    "System Downtime": "Root cause identified and resolved by IT; monitoring added to detect recurrence.",
    "Process Deviation": "Process step re-applied retroactively; refresher briefing given to the team.",
    "Compliance Issue": "Missing documentation obtained and filed; case closed following compliance sign-off.",
    "Billing Discrepancy": "Billing record corrected and reissued; reconciliation completed with client.",
    "Vendor Delay": "Vendor notified; revised SLA checkpoint agreed to prevent future delays.",
}


def random_date_within(days_back: int) -> datetime:
    """Random datetime within the last `days_back` days."""
    delta = timedelta(
        days=random.randint(0, days_back),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
    )
    return datetime.now() - delta

def generate_rows(n: int) -> pd.DataFrame:
    rows = []
    for i in range(1, n + 1):
        created_at = random_date_within(730)  
        category = random.choice(CATEGORIES)
        severity = random.choices(SEVERITIES, weights=SEVERITY_WEIGHTS, k=1)[0]
        owner = random.choice(OWNERS)

        if severity == "High":
            status = random.choices(
                ["Pending", "Approved", "Rejected", "Closed/Resolved"],
                weights=[0.10, 0.20, 0.10, 0.60],
                k=1,
            )[0]
        else:
            status = random.choices(
                ["Pending", "Closed/Resolved"],
                weights=[0.15, 0.85],
                k=1,
            )[0]

        approver = None
        approver_comments = None
        resolved_at = None

        went_through_approval = severity == "High" and status in (
            "Approved",
            "Rejected",
            "Closed/Resolved",
        )

        if went_through_approval:
            approver = random.choice(APPROVERS)
            if status == "Rejected":
                approver_comments = random.choice(APPROVER_COMMENTS_REJECTED)
            else:
                approver_comments = random.choice(APPROVER_COMMENTS_APPROVED)

        if status == "Closed/Resolved":
            resolved_at = created_at + timedelta(hours=random.randint(2, 96))
            resolution_note = RESOLUTION_NOTES_BY_CATEGORY[category]
        else:
            resolution_note = None

        description = random.choice(DESCRIPTIONS_BY_CATEGORY[category])

        exception_ref = f"EXC-{created_at.year}-{i:05d}"

        rows.append(
            {
                "ExceptionID": i,
                "ExceptionRef": exception_ref,
                "DateCreated": created_at,
                "Category": category,
                "Severity": severity,
                "Owner": owner,
                "Description": description,
                "Status": status,
                "Approver": approver,
                "Approver_Comments": approver_comments,
                "Resolution": resolution_note,
                "DateResolved": resolved_at,
            }
        )

    df = pd.DataFrame(rows)

    df["ResolutionTimeHours"] = (
        (df["DateResolved"] - df["DateCreated"]).dt.total_seconds() / 3600
    ).round(1)

    return df

def build_connection_string() -> str:
    driver = "ODBC Driver 17 for SQL Server"

    if USE_WINDOWS_AUTH:
        params = urllib.parse.quote_plus(
            f"DRIVER={{{driver}}};"
            f"SERVER={SQL_SERVER};"
            f"DATABASE={DATABASE};"
            f"Trusted_Connection=yes;"
        )
    else:
        params = urllib.parse.quote_plus(
            f"DRIVER={{{driver}}};"
            f"SERVER={SQL_SERVER};"
            f"DATABASE={DATABASE};"
            f"UID={SQL_USERNAME};"
            f"PWD={SQL_PASSWORD};"
        )

    return f"mssql+pyodbc:///?odbc_connect={params}"

def push_to_sql(df: pd.DataFrame):
    engine = create_engine(build_connection_string())

    df.to_sql(
        TABLE_NAME,
        con=engine,
        if_exists="replace",  
        index=False,
        dtype=None,  
    print(f"Loaded {len(df)} rows into {DATABASE}.dbo.{TABLE_NAME}")


if __name__ == "__main__":
    print(f"Generating {ROW_COUNT} mock exception records...")
    data = generate_rows(ROW_COUNT)

    print("\nSample rows:")
    print(data.head(5).to_string(index=False))

    print("\nStatus distribution:")
    print(data["Status"].value_counts())

    print("\nPushing to SQL Server...")
    push_to_sql(data)

Generating 2000 mock exception records...

Sample rows:
 ExceptionID   ExceptionRef                DateCreated            Category Severity           Owner                                                                      Description          Status     Approver                             Approver_Comments                                                                           Resolution               DateResolved  ResolutionTimeHours
           1 EXC-2024-00001 2024-12-11 13:45:10.736014 Billing Discrepancy      Low   Daniel Wagner    Invoice amount did not match the agreed contract rate for the billing period.         Pending          NaN                                           NaN                                                                                  NaN                        NaT                  NaN
           2 EXC-2024-00002 2024-11-02 16:49:10.736076    Compliance Issue      Low Abigail Shaffer              Required documentation was missing from the client on